# Benchmark: highspy vía LP (solo)

Mide tiempos del flujo productivo **write LP → highspy.readModel → run → mapeo a Pyomo**.

**Datos:** CSVs del escenario regional; años en `BENCHMARK_YEARS`.

**Experimento:** `run_crossover=choose` + barrido de tolerancias IPM (celda 4) para alinear IPM/postsolve en **una sola pasada**.

**Hilos** (celda 1, `BENCHMARK_THREADS`):
- `0` → todos los CPUs
- `N > 0` → N hilos fijos
- `None` → `SIM_SOLVER_THREADS`

**Kernel:** `backend/.venv`

In [1]:
# Celda 1 — Setup
from __future__ import annotations

import os
import shutil
import sys
import tempfile
import zipfile
from pathlib import Path
from time import perf_counter

import pandas as pd
import pyomo.environ as pyo
from IPython.display import display
from pyomo.core import Var

import highspy

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "backend" / "app").is_dir():
    BACKEND_ROOT = REPO_ROOT / "backend"
elif (REPO_ROOT.parent / "backend" / "app").is_dir():
    REPO_ROOT = REPO_ROOT.parent
    BACKEND_ROOT = REPO_ROOT / "backend"
else:
    raise RuntimeError(
        "Ejecuta el notebook desde la raíz del repo o desde notebooks/"
    )

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.simulation.core.data_processing import (
    eliminar_valores_fuera_de_indices,
    get_processing_result_from_csv_dir,
    normalize_mode_of_operation_in_csv_dir,
    reorder_activity_ratio_csvs_for_dataportal,
    strip_whitespace_in_set_csvs,
)
from app.simulation.core.instance_builder import build_instance
from app.simulation.core.model_definition import create_abstract_model
from app.simulation.core.solver import (
    _effective_solver_threads,
    _hardware_thread_limit,
    _reset_highspy_scheduler,
)

CSV_ZIP = Path(
    "/home/jchavez/Documentos/UPME/Datos Simulacion/Regional/Caso 1/CSV.zip"
)
BENCHMARK_YEARS = {2022, 2023, 2024, 2025}
WORK_DIR = Path(tempfile.mkdtemp(prefix="osemosys_highspy_lp_"))
CSV_DIR = WORK_DIR / "csv"
LP_PATH = WORK_DIR / "model.lp"

# Opciones HiGHS base (alineadas con solver.py tras barrido celda 4)
HIGHS_SOLVER_METHOD = "ipm"
HIGHS_PRESOLVE = "on"
HIGHS_PARALLEL = "on"
HIGHS_RUN_CROSSOVER = "choose"
HIGHS_IPM_OPTIMALITY_TOLERANCE = 1e-12  # ganador barrido regional (~11s, kOptimal)
HIGHS_EXTRA_OPTIONS: dict[str, float | int | str] = {
    "ipm_optimality_tolerance": HIGHS_IPM_OPTIMALITY_TOLERANCE,
}

# --- Hilos: 0 = auto (cap hardware); N>0 se recorta a cpus disponibles ---
BENCHMARK_THREADS: int | None = 0
_threads_config = (
    BENCHMARK_THREADS
    if BENCHMARK_THREADS is not None
    else int(os.getenv("SIM_SOLVER_THREADS", "0") or 0)
)
_threads_configured = _threads_config
SOLVER_THREADS_EFFECTIVE = _effective_solver_threads(_threads_config)
if _threads_config > 0:
    SOLVER_THREADS_LABEL = str(_threads_config)
else:
    SOLVER_THREADS_LABEL = f"all({SOLVER_THREADS_EFFECTIVE})"

print("REPO_ROOT:", REPO_ROOT)
print("WORK_DIR:", WORK_DIR)
print(
    f"Hilos: config={_threads_configured!r} → aplicados={SOLVER_THREADS_EFFECTIVE} "
    f"(hardware={_hardware_thread_limit()}, cpus={os.cpu_count()})"
)

REPO_ROOT: /home/jchavez/Documentos/APPS/UPME/Osemosys_UPME
WORK_DIR: /tmp/osemosys_highspy_lp_2qhq_fg6
Hilos: config=0 → aplicados=16 (hardware=16, cpus=16)


In [2]:
# Celda 2 — CSVs + instancia Pyomo

def trim_csvs_to_years(csv_dir: Path, keep_years: set[int]) -> None:
    year_csv = csv_dir / "YEAR.csv"
    if year_csv.is_file():
        df = pd.read_csv(year_csv)
        col = df.columns[0]
        df = df[df[col].astype(int).isin(keep_years)]
        df.to_csv(year_csv, index=False)
    for csv_file in csv_dir.glob("*.csv"):
        if csv_file.name == "YEAR.csv":
            continue
        df = pd.read_csv(csv_file, low_memory=False)
        if "YEAR" in df.columns:
            df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")
            df = df[df["YEAR"].isin(keep_years)]
            df.to_csv(csv_file, index=False)


def unzip_csvs(zip_path: Path, dest_csv_dir: Path) -> Path:
    if not zip_path.is_file():
        raise FileNotFoundError(f"No existe: {zip_path}")
    dest_csv_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_csv_dir.parent)
    nested = dest_csv_dir.parent / "CSV"
    if nested.is_dir() and not any(dest_csv_dir.iterdir()):
        for item in nested.iterdir():
            shutil.move(str(item), str(dest_csv_dir / item.name))
        nested.rmdir()
    return dest_csv_dir


def preprocess_csv_dir(csv_dir: Path) -> None:
    reorder_activity_ratio_csvs_for_dataportal(str(csv_dir))
    normalize_mode_of_operation_in_csv_dir(str(csv_dir))
    strip_whitespace_in_set_csvs(str(csv_dir))
    eliminar_valores_fuera_de_indices(str(csv_dir))


def build_concrete_instance(csv_dir: Path):
    preprocess_csv_dir(csv_dir)
    proc = get_processing_result_from_csv_dir(str(csv_dir))
    has_storage = proc.has_storage
    has_udc = proc.has_udc
    print(f"has_storage={has_storage}, has_udc={has_udc}")
    print(
        f"REGION={len(proc.sets.get('REGION', []))}, "
        f"TECH={len(proc.sets.get('TECHNOLOGY', []))}, "
        f"YEAR={len(proc.sets.get('YEAR', []))}, "
        f"TS={len(proc.sets.get('TIMESLICE', []))}"
    )
    t0 = perf_counter()
    abstract = create_abstract_model(has_storage=has_storage, has_udc=has_udc)
    instance = build_instance(
        abstract,
        str(csv_dir),
        has_storage=has_storage,
        has_udc=has_udc,
    )
    return instance, perf_counter() - t0


csv_dir = unzip_csvs(CSV_ZIP, CSV_DIR)
trim_csvs_to_years(csv_dir, BENCHMARK_YEARS)
print("Años:", sorted(BENCHMARK_YEARS))

instance, build_seconds = build_concrete_instance(csv_dir)
print(f"Instancia construida en {build_seconds:.2f} s")

Años: [2022, 2023, 2024, 2025]
has_storage=True, has_udc=False
REGION=1, TECH=2343, YEAR=4, TS=1
           0 seconds to construct Set YEAR; 1 index total
           0 seconds to construct Set TECHNOLOGY; 1 index total
           0 seconds to construct Set TIMESLICE; 1 index total
           0 seconds to construct Set FUEL; 1 index total
           0 seconds to construct Set EMISSION; 1 index total
           0 seconds to construct Set MODE_OF_OPERATION; 1 index total
           0 seconds to construct Set REGION; 1 index total
           0 seconds to construct Set STORAGE; 1 index total
           0 seconds to construct Set SEASON; 1 index total
           0 seconds to construct Set DAYTYPE; 1 index total
           0 seconds to construct Set DAILYTIMEBRACKET; 1 index total
           0 seconds to construct Set STORAGEINTRADAY; 1 index total
           0 seconds to construct Set STORAGEINTRAYEAR; 1 index total
           0 seconds to construct Set FLEXIBLEDEMANDTYPE; 1 index total
    

In [3]:
# Celda 3 — highspy vía LP (write + read + solve + mapeo primals)


def pyomo_name_to_lp(name: str) -> str:
    if "[" in name and name.endswith("]"):
        base, rest = name.split("[", 1)
        return f"{base}({rest[:-1]})"
    return name


def lp_name_to_pyomo(name: str) -> str:
    if "(" in name and name.endswith(")"):
        base, rest = name.split("(", 1)
        return f"{base}[{rest[:-1]}]"
    return name


def highs_status_label(status: object) -> str:
    mapping = {
        getattr(highspy.HighsModelStatus, "kOptimal", None): "optimal",
        getattr(highspy.HighsModelStatus, "kInfeasible", None): "infeasible",
        getattr(highspy.HighsModelStatus, "kUnbounded", None): "unbounded",
    }
    for hs, label in mapping.items():
        if hs is not None and status == hs:
            return label
    return str(status)


def apply_highspy_solution(instance, h: highspy.Highs) -> float:
    solution = h.getSolution()
    lp = h.getLp()
    col_names = list(getattr(lp, "col_names_", []) or [])
    col_values = list(getattr(solution, "col_value", []) or [])
    col_map: dict[str, float] = {}
    for idx, name in enumerate(col_names):
        if idx < len(col_values):
            val = float(col_values[idx])
            col_map[name] = val
            col_map[lp_name_to_pyomo(name)] = val
    for var in instance.component_data_objects(Var, active=True):
        pyomo_name = var.name
        lp_name = pyomo_name_to_lp(pyomo_name)
        val = col_map.get(pyomo_name) or col_map.get(lp_name)
        if val is not None:
            var.set_value(val, skip_validation=True)
    try:
        return float(h.getInfo().objective_function_value)
    except Exception:
        return float(pyo.value(instance.OBJ))


def run_highspy_via_lp(
    instance,
    lp_path: Path,
    *,
    threads: int,
    solver_method: str = HIGHS_SOLVER_METHOD,
    presolve: str = HIGHS_PRESOLVE,
    parallel: str = HIGHS_PARALLEL,
    run_crossover: str = HIGHS_RUN_CROSSOVER,
    extra_options: dict | None = None,
) -> dict:
    lp_path = Path(lp_path)
    lp_path.parent.mkdir(parents=True, exist_ok=True)

    t_write = perf_counter()
    instance.write(
        filename=str(lp_path),
        io_options={"symbolic_solver_labels": True},
    )
    write_lp_seconds = perf_counter() - t_write
    lp_size_mb = lp_path.stat().st_size / (1024 * 1024)

    h = highspy.Highs()
    _reset_highspy_scheduler()
    h.setOptionValue("log_to_console", False)
    h.setOptionValue("output_flag", False)
    h.setOptionValue("solver", solver_method)
    h.setOptionValue("presolve", presolve)
    h.setOptionValue("parallel", parallel)
    h.setOptionValue("run_crossover", run_crossover)
    h.setOptionValue("threads", threads)
    for key, val in (extra_options or HIGHS_EXTRA_OPTIONS).items():
        h.setOptionValue(key, val)

    t_read = perf_counter()
    h.readModel(str(lp_path))
    read_model_seconds = perf_counter() - t_read

    t_run = perf_counter()
    h.run()
    run_seconds = perf_counter() - t_run

    status = highs_status_label(h.getModelStatus())
    obj = 0.0
    map_seconds = 0.0
    if "optimal" in status.lower():
        t_map = perf_counter()
        obj = apply_highspy_solution(instance, h)
        map_seconds = perf_counter() - t_map

    return {
        "status": status,
        "objective": obj,
        "build_seconds": build_seconds,
        "write_lp_seconds": write_lp_seconds,
        "read_model_seconds": read_model_seconds,
        "run_seconds": run_seconds,
        "map_solution_seconds": map_seconds,
        "total_seconds": (
            build_seconds
            + write_lp_seconds
            + read_model_seconds
            + run_seconds
            + map_seconds
        ),
        "solve_pipeline_seconds": (
            write_lp_seconds + read_model_seconds + run_seconds + map_seconds
        ),
        "lp_size_mb": lp_size_mb,
        "solver_method": solver_method,
        "presolve": presolve,
        "parallel": parallel,
        "run_crossover": run_crossover,
        "extra_options": dict(extra_options or HIGHS_EXTRA_OPTIONS),
        "threads_config": SOLVER_THREADS_LABEL,
        "threads_effective": threads,
    }


result = run_highspy_via_lp(
    instance,
    LP_PATH,
    threads=SOLVER_THREADS_EFFECTIVE,
)
print("highspy via LP:", result)
print(f"LP: {LP_PATH} ({result['lp_size_mb']:.2f} MB)")

highspy via LP: {'status': 'optimal', 'objective': 139114.2969342994, 'build_seconds': 13.178158581999014, 'write_lp_seconds': 25.239833612999064, 'read_model_seconds': 6.404622679001477, 'run_seconds': 12.315716779998183, 'map_solution_seconds': 7.635190209999564, 'total_seconds': 64.7735218639973, 'solve_pipeline_seconds': 51.59536328199829, 'lp_size_mb': 110.83116340637207, 'solver_method': 'ipm', 'presolve': 'on', 'parallel': 'on', 'run_crossover': 'choose', 'extra_options': {'ipm_optimality_tolerance': 1e-12}, 'threads_config': 'all(16)', 'threads_effective': 16}
LP: /tmp/osemosys_highspy_lp_2qhq_fg6/model.lp (110.83 MB)


In [4]:
# Celda 4 — Barrido tolerancias IPM (choose, una pasada/config; ~5-12s c/u)
# Referencia: crossover=on ~26s regional. kkt_* omitido (puede tardar >60s).

IPM_TOLERANCE_SWEEP = [
    ("baseline", {}),
    ("ipm_1e-10", {"ipm_optimality_tolerance": 1e-10}),
    ("ipm_1e-12", {"ipm_optimality_tolerance": 1e-12}),
    ("cross_1e-10", {"start_crossover_tolerance": 1e-10}),
    ("cross_1e-12", {"start_crossover_tolerance": 1e-12}),
]


if not LP_PATH.is_file():
    raise FileNotFoundError(
        f"No existe {LP_PATH}. Ejecuta las celdas 1→2→3 antes del barrido."
    )


def run_tolerance_case(label: str, extra: dict) -> dict:
    opts = dict(extra)
    _reset_highspy_scheduler()
    h = highspy.Highs()
    h.setOptionValue("log_to_console", False)
    h.setOptionValue("output_flag", False)
    h.setOptionValue("solver", "ipm")
    h.setOptionValue("presolve", "on")
    h.setOptionValue("parallel", "on")
    h.setOptionValue("run_crossover", opts.pop("run_crossover", HIGHS_RUN_CROSSOVER))
    h.setOptionValue("threads", SOLVER_THREADS_EFFECTIVE)
    for key, val in opts.items():
        h.setOptionValue(key, val)
    t0 = perf_counter()
    h.readModel(str(LP_PATH))
    read_s = perf_counter() - t0
    t1 = perf_counter()
    h.run()
    run_s = perf_counter() - t1
    info = h.getInfo()
    status = h.getModelStatus()
    max_di = float(getattr(info, "max_dual_infeasibility", 0.0))
    return {
        "label": label,
        "status": str(status),
        "optimal": "Optimal" in str(status),
        "objective": float(getattr(info, "objective_function_value", 0.0)),
        "max_primal_infeasibility": float(getattr(info, "max_primal_infeasibility", 0.0)),
        "max_dual_infeasibility": max_di,
        "dual_ok": max_di < 1e-7,
        "read_seconds": read_s,
        "run_seconds": run_s,
        "total_seconds": read_s + run_s,
        "extra": dict(extra),
    }


sweep_rows = [run_tolerance_case(label, extra) for label, extra in IPM_TOLERANCE_SWEEP]
sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df)

winners = sweep_df[
    sweep_df["optimal"]
    & sweep_df["dual_ok"]
    & (sweep_df["total_seconds"] <= 30.0)
]
if not winners.empty:
    winner = winners.sort_values("total_seconds").iloc[0]
    print(
        f"Ganador: {winner['label']} extra={winner['extra']} "
        f"obj={winner['objective']:.2f} total={winner['total_seconds']:.1f}s "
        f"(ref crossover=on ~26s)"
    )
    if winner["label"] == "ipm_1e-12":
        print("→ Aplicado en solver.py: run_crossover=choose + ipm_optimality_tolerance=1e-12")
else:
    print("Sin ganador: fallback retry crossover=on solo si Unknown (ver solver.py)")

,label,status,optimal,objective,max_primal_infeasibility,max_dual_infeasibility,dual_ok,read_seconds,run_seconds,total_seconds,extra
0,baseline,HighsModelStatus.kUnknown,False,139114.296934,1.655253e-08,1.640877e+03,False,5.973613,12.672408,18.646022,{}
1,ipm_1e-10,HighsModelStatus.kUnknown,False,139114.296934,1.655253e-08,1.640877e+03,False,5.551185,10.951339,16.502524,{'ipm_optimality_tolerance': 1e-10}
2,ipm_1e-12,HighsModelStatus.kOptimal,True,139114.296934,1.888980e-08,5.866368e-08,True,5.679535,12.577172,18.256707,{'ipm_optimality_tolerance': 1e-12}
3,cross_1e-10,HighsModelStatus.kUnknown,False,139114.296934,1.655184e-08,1.640877e+03,False,5.850238,10.620873,16.471111,{'start_crossover_tolerance': 1e-10}
4,cross_1e-12,HighsModelStatus.kUnknown,False,139114.296934,1.655184e-08,1.640877e+03,False,5.659832,11.350762,17.010593,{'start_crossover_tolerance': 1e-12}


Ganador: ipm_1e-12 extra={'ipm_optimality_tolerance': 1e-12} obj=139114.30 total=18.3s (ref crossover=on ~26s)
→ Aplicado en solver.py: run_crossover=choose + ipm_optimality_tolerance=1e-12


In [5]:
# Celda 5 — Resumen de tiempos

df = pd.DataFrame(
    [
        {"fase": "build_instance", "segundos": result["build_seconds"]},
        {"fase": "write_lp", "segundos": result["write_lp_seconds"]},
        {"fase": "read_model", "segundos": result["read_model_seconds"]},
        {"fase": "highs_run", "segundos": result["run_seconds"]},
        {"fase": "map_solution", "segundos": result["map_solution_seconds"]},
        {"fase": "solve_pipeline (sin build)", "segundos": result["solve_pipeline_seconds"]},
        {"fase": "TOTAL (build + solve)", "segundos": result["total_seconds"]},
    ]
)
display(df)
print(
    f"status={result['status']}, objective={result['objective']:.4f}, "
    f"hilos={result['threads_config']}"
)

,fase,segundos
0,build_instance,13.178159
1,write_lp,25.239834
2,read_model,6.404623
3,highs_run,12.315717
4,map_solution,7.635190
5,solve_pipeline (sin build),51.595363
6,TOTAL (build + solve),64.773522


status=optimal, objective=139114.2969, hilos=all(16)


In [6]:
# Celda 5 (opcional) — Barrido de hilos reutilizando el .lp ya escrito
# Solo readModel + run (sin rewrite LP ni rebuild). Puede tardar varios minutos.

THREAD_SWEEP = False  # True para comparar 1 vs 4 vs all(cpus)

if not THREAD_SWEEP:
    print("THREAD_SWEEP=False — omitido.")
elif not LP_PATH.is_file():
    raise FileNotFoundError("Ejecuta la celda 3 primero.")
else:
    cpu_n = os.cpu_count() or 1
    sweep_configs = [
        (1, "1"),
        (4, "4"),
        (_effective_solver_threads(0), f"all({cpu_n})"),
    ]
    sweep_rows = []
    for threads_n, label in sweep_configs:
        h = highspy.Highs()
        h.setOptionValue("log_to_console", False)
        h.setOptionValue("output_flag", False)
        h.setOptionValue("solver", HIGHS_SOLVER_METHOD)
        h.setOptionValue("presolve", HIGHS_PRESOLVE)
        h.setOptionValue("parallel", HIGHS_PARALLEL)
        h.setOptionValue("run_crossover", HIGHS_RUN_CROSSOVER)
        h.setOptionValue("threads", threads_n)
        t0 = perf_counter()
        h.readModel(str(LP_PATH))
        read_s = perf_counter() - t0
        t1 = perf_counter()
        h.run()
        run_s = perf_counter() - t1
        sweep_rows.append({
            "hilos": label,
            "threads_effective": threads_n,
            "read_model_seconds": read_s,
            "run_seconds": run_s,
            "total_seconds": read_s + run_s,
            "status": highs_status_label(h.getModelStatus()),
        })
    display(pd.DataFrame(sweep_rows))

THREAD_SWEEP=False — omitido.


In [7]:
# Opcional: limpiar WORK_DIR
# shutil.rmtree(WORK_DIR, ignore_errors=True)